In [ ]:
%pip -q install datasets scikit-learn

# Bài 14: So sánh Transformer khởi tạo ngẫu nhiên và pretrained fine-tuning
import random, time
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
MODEL_NAME = "bert-base-uncased"
TRAIN_SIZE, VALID_SIZE, TEST_SIZE = 256, 128, 256
MAX_LENGTH, EPOCHS, BATCH_SIZE = 64, 1, 8
LEARNING_RATE = 2e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading IMDb and preparing identical splits...")
dataset = load_dataset("stanfordnlp/imdb")
train_val = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE + VALID_SIZE))
train_raw = train_val.select(range(TRAIN_SIZE))
valid_raw = train_val.select(range(TRAIN_SIZE, TRAIN_SIZE + VALID_SIZE))
test_raw = dataset["test"].shuffle(seed=SEED + 1).select(range(TEST_SIZE))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

def prepare_split(split):
    encoded = split.map(tokenize_batch, batched=True, remove_columns=["text"])
    encoded = encoded.rename_column("label", "labels")
    encoded.set_format(type="torch")
    return encoded

train_data = prepare_split(train_raw)
valid_data = prepare_split(valid_raw)
test_data = prepare_split(test_raw)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
valid_loader = DataLoader(valid_data, batch_size=16, shuffle=False)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
def create_model(kind):
    torch.manual_seed(SEED)
    if kind == "scratch":
        return AutoModelForSequenceClassification.from_config(config)
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def evaluate(model, loader):
    model.eval()
    all_labels, all_predictions = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            logits = model(**batch).logits
            all_predictions.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(batch["labels"].cpu().tolist())
    return accuracy_score(all_labels, all_predictions), f1_score(all_labels, all_predictions, average="binary")

def run_experiment(kind):
    model = create_model(kind).to(device)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    torch.manual_seed(SEED)
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    model.train()
    start = time.perf_counter()
    for epoch in range(EPOCHS):
        total_loss = 0.0
        for step, batch in enumerate(train_loader, start=1):
            batch = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            output = model(**batch)
            output.loss.backward()
            optimizer.step()
            total_loss += output.loss.item()
            if step % 10 == 0:
                print(f"{kind}: epoch {epoch + 1}/{EPOCHS}, batch {step}/{len(train_loader)}")
    training_seconds = time.perf_counter() - start
    valid_accuracy, valid_f1 = evaluate(model, valid_loader)
    test_accuracy, test_f1 = evaluate(model, test_loader)
    result = {
        "name": "Transformer from scratch" if kind == "scratch" else "Pretrained Transformer",
        "accuracy": test_accuracy, "f1": test_f1,
        "validation_accuracy": valid_accuracy, "validation_f1": valid_f1,
        "training_seconds": training_seconds, "parameters": parameter_count,
        "mean_train_loss": total_loss / len(train_loader),
    }
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

print(f"Device: {device}; model: {MODEL_NAME}; train/validation/test: {TRAIN_SIZE}/{VALID_SIZE}/{TEST_SIZE}; max_length={MAX_LENGTH}; epochs={EPOCHS}; batch_size={BATCH_SIZE}")
results = [run_experiment("scratch"), run_experiment("pretrained")]
print("\nFINAL RESULTS (test set)")
for result in results:
    print(f"{result['name']}: Accuracy={result['accuracy']:.4f}, F1={result['f1']:.4f}, training_time={result['training_seconds']:.2f}s, parameters={result['parameters']:,}")
    print(f"  Validation: Accuracy={result['validation_accuracy']:.4f}, F1={result['validation_f1']:.4f}; mean training loss={result['mean_train_loss']:.4f}")
print("\nControlled setup: same sampled IMDb splits, tokenizer, architecture, sequence length, batch size, optimizer, learning rate, and epoch count. Small subset/one epoch is for a resource-limited comparison, not a benchmark.")

Loading IMDb and preparing identical splits...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


pretrained: epoch 1/1, batch 10/32
pretrained: epoch 1/1, batch 20/32
pretrained: epoch 1/1, batch 30/32

FINAL RESULTS (test set)
Transformer from scratch: Accuracy=0.5430, F1=0.0000, training_time=4.45s, parameters=109,483,778
  Validation: Accuracy=0.4609, F1=0.0000; mean training loss=0.7650
Pretrained Transformer: Accuracy=0.6914, F1=0.5907, training_time=3.61s, parameters=109,483,778
  Validation: Accuracy=0.6719, F1=0.6038; mean training loss=0.6705

Controlled setup: same sampled IMDb splits, tokenizer, architecture, sequence length, batch size, optimizer, learning rate, and epoch count. Small subset/one epoch is for a resource-limited comparison, not a benchmark.
